In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
## Create a lookup table
doc_idx = {}

for doc in documents:
    doc_idx[doc['id']] = doc

## Running RAG

In [5]:
## Import as usual
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [ ]:
## Ask LLM to answer
rec = ground_truth[0]
question = rec['question']

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [8]:
assistant.total_cost()

0.00058575

In [ ]:
## Get the original answer
doc_id = rec['document']
original_doc = doc_idx[doc_id]
answer_orig = original_doc['answer']
answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [14]:
## Save both answers in one record
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course late — am I still allowed to join in now?',
 'answer_llm': 'Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

## Processing all questions

In [16]:
## Create function processes one ground truth record
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [17]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course late — am I still allowed to join in now?',
 'answer_llm': 'Yes — you can still join now. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [18]:
## Before running full batch, reset the usage
assistant.reset_usage()

In [19]:
## It calls LLM once per ground truth question
## Process the questions in parallel and track progress
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [21]:
## Run RAG for all ground truth questions
with ThreadPoolExecutor(max_workers=3) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/590 [00:00<?, ?it/s]

In [22]:
## Collect answer record
answers = []

for answer_record in results:
    answers.append(answer_record)

In [23]:
## Calculate the cost
assistant.total_cost()

1.2876712500000007

In [24]:
## Save answers:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [25]:
df_answers.head()

,question,answer_llm,answer_orig,document
0,I just found this course late — am I still all...,Yes — you can still join. If you want a certif...,"Yes, but if you want to receive a certificate,...",74eb249bbf
1,Is it too late to start the course if I missed...,"No, it’s not too late. You can start whenever ...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,Can I enroll after the course has already star...,I don't know.,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,"If I join the course now, can I still get a ce...","Yes, but only if you submit your project while...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What’s the deadline for getting the certificat...,I don’t know.,"Yes, but if you want to receive a certificate,...",74eb249bbf
